In [1]:
QUERY_AUDIO_PATH = "C:/Users/marti/Desktop/dark-distorted-hard-trap-bass_F_minor.wav"

FILENAME= QUERY_AUDIO_PATH.split("/")[-1].split(".")[0]
FILENAME

'dark-distorted-hard-trap-bass_F_minor'

In [2]:
import asyncio
import json
import sys
import os
from openai import OpenAI
from qdrant_client import QdrantClient

# Setup Path
sys.path.append(os.path.abspath(".."))
from config.settings import settings
from rag.clap.model_handler import create_clap_model

# --- CONFIGURAZIONE ---
QDRANT_COLLECTION = settings.QDRANT_ENRICHED_COLLECTION_NAME
# Scegli un file audio rappresentativo che hai nel DB
TEST_AUDIO_FILE = QUERY_AUDIO_PATH
# Una query di ricerca interessante
TEST_QUERY = "Dark cinematic atmosphere with granular textures"

def print_section(title):
    print("\n" + "="*60)
    print(f" {title}")
    print("="*60 + "\n")

async def run_end_to_end_demo():
    client_q = QdrantClient(host=settings.QDRANT_CONNECTION_HOST, port=settings.QDRANT_PORT)
    client_o = OpenAI(api_key=settings.MODEL_API_KEY)
    clap = create_clap_model()

    # =================================================================
    # SCENARIO A: ANALYSIS FLOW (Audio -> Analysis -> Explanation)
    # =================================================================
    print_section("SCENARIO A: AUDIO ANALYSIS FLOW (Reverse-RAG)")

    if not os.path.exists(TEST_AUDIO_FILE):
        print(f"⚠️ File audio '{TEST_AUDIO_FILE}' non trovato. Salto analisi audio.")
    else:
        print(f"INPUT: File Audio '{TEST_AUDIO_FILE}'")

        # 1. Perception (CLAP)
        print("\n--- STEP 1: PERCEPTION (Audio Embedding) ---")
        audio_vec = clap.get_audio_embedding([TEST_AUDIO_FILE])[0]
        print(f"Generated Vector: [ {audio_vec[0]:.4f}, {audio_vec[1]:.4f}, ... ] (Size: 512)")

        # 2. Retrieval (Nearest Neighbors)
        print("\n--- STEP 2: KNOWLEDGE RETRIEVAL (Qdrant) ---")
        search_result = client_q.query_points(
            collection_name=QDRANT_COLLECTION,
            query=audio_vec,
            using="audio_vector",
            limit=3,
            with_payload=True
        )

        neighbors_data = []
        for hit in search_result.points:
            p = hit.payload
            neighbors_data.append({
                "filename": p.get('original_filename'),
                "ai_description": p.get('ai_label'),
                "similarity": round(hit.score, 4),
                "tags": p.get('ai_tags', [])[:3]
            })

        print(json.dumps(neighbors_data, indent=2))

        # 3. Reasoning (LLM Synthesis)
        print("\n--- STEP 3: AGENT REASONING (Prompting) ---")
        context_str = "\n".join([f"- Sample '{n['filename']}' ({n['similarity']} similarity): {n['ai_description']}" for n in neighbors_data])

        prompt = f"""
        ANALYZE this unknown audio file based on its nearest neighbors in the database:
        {context_str}

        TASK: Synthesize a definitive description for the input file.
        CONSTRAINT: Ground your answer strictly in the provided neighbors.
        """
        print(f"System Prompt Payload:\n{prompt.strip()}")

        # 4. Output (Simulazione Humanizer)
        print("\n--- STEP 4: FINAL HUMANIZED OUTPUT ---")
        # Qui in realtà chiameresti il tuo AudioAnalystAgent, simuliamo la risposta per la demo
        print(f"> \"Analyzing the acoustic features, this sample appears to be a '{neighbors_data[0]['ai_description']}'. It shares strong timbral similarities ({neighbors_data[0]['similarity']}) with established granular textures in the library, confirming its classification as a dark cinematic element.\"")


    # =================================================================
    # SCENARIO B: RETRIEVAL FLOW (Text -> Results -> Human Response)
    # =================================================================
    print_section("SCENARIO B: RETRIEVAL FLOW (Standard RAG)")
    print(f"INPUT QUERY: \"{TEST_QUERY}\"")

    # 1. Understanding (Embedding)
    print("\n--- STEP 1: QUERY EMBEDDING ---")
    emb_res = client_o.embeddings.create(input=TEST_QUERY, model=settings.MODEL_EMBEDDING_MODEL)
    query_vec = emb_res.data[0].embedding
    print(f"Vectorized Query: [ {query_vec[0]:.4f}, ... ] (Size: 1536)")

    # 2. Retrieval
    print("\n--- STEP 2: VECTOR SEARCH ---")
    search_res = client_q.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vec,
        using="text_vector",
        limit=3,
        with_payload=True
    )

    results_json = []
    for hit in search_res.points:
        results_json.append({
            "id": hit.payload.get('original_filename'),
            "score": round(hit.score, 3),
            "desc": hit.payload.get('ai_label')
        })
    print(json.dumps(results_json, indent=2))

    # 3. Humanizer
    print("\n--- STEP 3: HUMANIZER AGENT OUTPUT ---")
    print(f"> \"I found 3 samples matching '{TEST_QUERY}'. The best match is '{results_json[0]['id']}' (Score: {results_json[0]['score']}), which features exactly the requested granular texture. Another valid option is '{results_json[1]['id']}' if you need a slightly different variation.\"")

if __name__ == "__main__":
    asyncio.run(run_end_to_end_demo())

H:\music-ai\venvmusic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: asyncio.run() cannot be called from a running event loop